STEP 1: Install Libraries

In [3]:
!pip install transformers torch langchain langchain-community

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


STEP 2: Imports

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate

STEP 3: Create LLM

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline 

pipe = pipeline(
    "text-generation",   # ✅ FIXED
    model="google/flan-t5-base",
    max_length=200
)

llm = HuggingFacePipeline(pipeline=pipe)

STEP 4: Create Prompts

In [5]:
from langchain_core.prompts import PromptTemplate

# Extract Prompt
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Experience
- Tools

Resume:
{resume}

Return JSON.
"""
)

# Match Prompt
match_prompt = PromptTemplate(
    input_variables=["resume_data", "job_desc"],
    template="""
Compare candidate with job.

Candidate:
{resume_data}

Job:
{job_desc}

Return matching and missing skills.
"""
)

# Score Prompt
score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
Give score (0–100).

Rules:
- More match → higher score
- No skills → below 30

{match_data}

Return only number.
"""
)

# Explain Prompt
explain_prompt = PromptTemplate(
    input_variables=["score", "match_data"],
    template="""
Explain score.

Score: {score}
Details: {match_data}
"""
)

STEP 5: Create Chains

In [6]:
extract_chain = extract_prompt | llm
match_chain = match_prompt | llm
score_chain = score_prompt | llm
explain_chain = explain_prompt | llm

STEP 6: Create Pipeline Function

In [7]:
def run_pipeline(resume, job_desc):

    extracted = extract_chain.invoke({"resume": resume})
    
    matched = match_chain.invoke({
        "resume_data": extracted,
        "job_desc": job_desc
    })
    
    score = score_chain.invoke({
        "match_data": matched
    })
    
    explanation = explain_chain.invoke({
        "score": score,
        "match_data": matched
    })
    
    return score, explanation

STEP 7: Input Data

In [8]:
job_desc = """
Data Scientist role:
- Python
- Machine Learning
- SQL
- Pandas, NumPy
"""

resume_strong = """
3 years Data Scientist
Skills: Python, Machine Learning, SQL, Pandas, NumPy
"""

resume_avg = """
Python developer
Skills: Python, SQL
"""

resume_weak = """
Fresher
Skills: HTML, C
"""

STEP 8: Run Loop

In [9]:
for name, resume in {
    "Strong": resume_strong,
    "Average": resume_avg,
    "Weak": resume_weak
}.items():
    
    score, explanation = run_pipeline(resume, job_desc)
    
    print(f"\n--- {name} Candidate ---")
    print("Score:", score)
    print("Explanation:", explanation)


--- Strong Candidate ---
Score: 
Give score (0–100).

Rules:
- More match → higher score
- No skills → below 30


Compare candidate with job.

Candidate:

Extract:
- Skills
- Experience
- Tools

Resume:

3 years Data Scientist
Skills: Python, Machine Learning, SQL, Pandas, NumPy


Return JSON.


Job:

Data Scientist role:
- Python
- Machine Learning
- SQL
- Pandas, NumPy


Return matching and missing skills.


Return only number.

Explanation: 
Explain score.

Score: 
Give score (0–100).

Rules:
- More match → higher score
- No skills → below 30


Compare candidate with job.

Candidate:

Extract:
- Skills
- Experience
- Tools

Resume:

3 years Data Scientist
Skills: Python, Machine Learning, SQL, Pandas, NumPy


Return JSON.


Job:

Data Scientist role:
- Python
- Machine Learning
- SQL
- Pandas, NumPy


Return matching and missing skills.


Return only number.

Details: 
Compare candidate with job.

Candidate:

Extract:
- Skills
- Experience
- Tools

Resume:

3 years Data Scientist
S